# Support Ticket Classification with DistilBERT

This is a simple, self-contained notebook for learning the complete Transformer workflow. Run the cells from top to bottom. It uses `AutoTokenizer` for tokenization and `AutoModelForSequenceClassification` for the model.

## 1. Install packages (run only if needed)

In [1]:
# Uncomment this line in a fresh environment.
# %pip install datasets transformers accelerate torch scikit-learn pandas scipy

## 2. Imports and settings

In [2]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from scipy.special import softmax
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = 'distilbert-base-uncased'
DATASET_NAME = 'bitext/Bitext-customer-support-llm-chatbot-training-dataset'
TEXT_COLUMN = 'instruction'
LABEL_COLUMN = 'intent'
MAX_LENGTH = 32  # Chosen from the earlier EDA; all cleaned tickets fit within 32 tokens.
OUTPUT_DIR = Path('models/supportticket-notebook-model')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')

c:\Users\harsh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## 3. Load the dataset

We use only the customer instruction as input and the intent as the target label.

In [3]:
raw_dataset = load_dataset(DATASET_NAME, split='train')
df = raw_dataset.to_pandas()[[TEXT_COLUMN, LABEL_COLUMN]]
print('Original shape:', df.shape)
display(df.head())
print('Number of intents:', df[LABEL_COLUMN].nunique())

Original shape: (26872, 2)


,instruction,intent
0,question about cancelling order {{Order Number}},cancel_order
1,i have a question about cancelling oorder {{Or...,cancel_order
2,i need help cancelling puchase {{Order Number}},cancel_order
3,I need to cancel purchase {{Order Number}},cancel_order
4,"I cannot afford this order, cancel purchase {{...",cancel_order


Number of intents: 27


## 4. Minimal cleaning

We remove missing, empty, and duplicate examples. We do not remove stopwords, punctuation, or apply stemming because the pretrained tokenizer expects natural text.

In [4]:
df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN]).copy()
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()
df[LABEL_COLUMN] = (
    df[LABEL_COLUMN].astype(str).str.strip().str.lower().str.replace(r'[^a-z0-9]+', '_', regex=True)
)
df = df[(df[TEXT_COLUMN] != '') & (df[LABEL_COLUMN] != '')]
df['_text_key'] = df[TEXT_COLUMN].str.casefold().str.replace(r'\s+', ' ', regex=True)

# Remove texts that appear with more than one intent, then remove duplicates.
label_counts = df.groupby('_text_key')[LABEL_COLUMN].nunique()
ambiguous_texts = set(label_counts[label_counts > 1].index)
df = df[~df['_text_key'].isin(ambiguous_texts)]
df = df.drop_duplicates('_text_key').drop(columns='_text_key').reset_index(drop=True)

print('Clean shape:', df.shape)
display(df[LABEL_COLUMN].value_counts().sort_index())

Clean shape: (24274, 2)


intent
cancel_order                436
change_order                856
change_shipping_address     961
check_cancellation_fee      941
check_invoice               913
check_payment_methods       993
check_refund_policy         974
complaint                   997
contact_customer_service    987
contact_human_agent         979
create_account              875
delete_account              901
delivery_options            655
delivery_period             990
edit_account                766
get_invoice                 888
get_refund                  911
newsletter_subscription     992
payment_issue               993
place_order                 992
recover_password            988
registration_problems       992
review                      993
set_up_shipping_address     990
switch_account              836
track_order                 783
track_refund                692
Name: count, dtype: int64

## 5. Create stratified train, validation, and test splits

In [5]:
train_df, remaining_df = train_test_split(
    df, train_size=0.70, random_state=SEED, stratify=df[LABEL_COLUMN]
)
validation_df, test_df = train_test_split(
    remaining_df, train_size=0.50, random_state=SEED, stratify=remaining_df[LABEL_COLUMN]
)
train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print('Train:', len(train_df))
print('Validation:', len(validation_df))
print('Test:', len(test_df))

Train: 16991
Validation: 3641
Test: 3642


## 6. Convert label names to IDs

The model needs numeric labels. We keep both mappings so predictions can be converted back to readable intent names.

In [6]:
label_names = sorted(train_df[LABEL_COLUMN].unique())
label2id = {label: index for index, label in enumerate(label_names)}
id2label = {index: label for label, index in label2id.items()}
print(label2id)

{'cancel_order': 0, 'change_order': 1, 'change_shipping_address': 2, 'check_cancellation_fee': 3, 'check_invoice': 4, 'check_payment_methods': 5, 'check_refund_policy': 6, 'complaint': 7, 'contact_customer_service': 8, 'contact_human_agent': 9, 'create_account': 10, 'delete_account': 11, 'delivery_options': 12, 'delivery_period': 13, 'edit_account': 14, 'get_invoice': 15, 'get_refund': 16, 'newsletter_subscription': 17, 'payment_issue': 18, 'place_order': 19, 'recover_password': 20, 'registration_problems': 21, 'review': 22, 'set_up_shipping_address': 23, 'switch_account': 24, 'track_order': 25, 'track_refund': 26}


## 7. Create the AutoTokenizer

`AutoTokenizer` automatically loads the correct tokenizer for DistilBERT. Token IDs are the numeric inputs. The attention mask marks real tokens with 1 and padding with 0.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
example = tokenizer(
    'My payment was deducted twice.',
    truncation=True,
    max_length=MAX_LENGTH,
)
print(example)
print(tokenizer.convert_ids_to_tokens(example['input_ids']))

{'input_ids': [101, 2026, 7909, 2001, 2139, 29510, 3807, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}
['[CLS]', 'my', 'payment', 'was', 'de', '##ducted', 'twice', '.', '[SEP]']


## 8. Convert pandas data into tokenized Hugging Face datasets

In [8]:
def make_huggingface_dataset(frame):
    model_frame = pd.DataFrame({
        TEXT_COLUMN: frame[TEXT_COLUMN],
        'labels': frame[LABEL_COLUMN].map(label2id),
    })
    dataset = Dataset.from_pandas(model_frame, preserve_index=False)

    def tokenize_batch(batch):
        return tokenizer(
            batch[TEXT_COLUMN],
            truncation=True,
            max_length=MAX_LENGTH,
        )

    return dataset.map(tokenize_batch, batched=True, remove_columns=[TEXT_COLUMN])

train_dataset = make_huggingface_dataset(train_df)
validation_dataset = make_huggingface_dataset(validation_df)
test_dataset = make_huggingface_dataset(test_df)
train_dataset[0]

Map: 100%|██████████| 3642/3642 [00:00<00:00, 30562.62 examples/s]


{'labels': 21,
 'input_ids': [101, 2393, 2000, 3189, 1037, 3291, 2007, 1037, 8819, 102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

## 9. Dynamic padding

The data collator pads each batch only to the longest sequence inside that batch.

In [9]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 10. Load DistilBERT with a classification head

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    label2id=label2id,
    id2label=id2label,
)
model.config.support_ticket_max_length = MAX_LENGTH

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1999.67it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 11. Define evaluation metrics

In [11]:
def compute_metrics(evaluation):
    logits, true_labels = evaluation
    predicted_labels = np.argmax(logits, axis=-1)
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        true_labels, predicted_labels, average='macro', zero_division=0
    )
    weighted_f1 = precision_recall_fscore_support(
        true_labels, predicted_labels, average='weighted', zero_division=0
    )[2]
    return {
        'accuracy': accuracy_score(true_labels, predicted_labels),
        'precision_macro': precision,
        'recall_macro': recall,
        'f1_macro': macro_f1,
        'f1_weighted': weighted_f1,
    }

## 12. Write TrainingArguments

These settings train for three epochs, evaluate and save after every epoch, and keep the checkpoint with the best validation macro F1.

In [12]:
training_args = TrainingArguments(
    output_dir='models/notebook-checkpoints',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=2,
    seed=SEED,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

## 13. Create the Trainer

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 14. Train the model

This is the long-running cell. On this machine, three CPU epochs took about one hour. A GPU will be much faster.

In [ ]:
train_result = trainer.train()
train_result

c:\Users\harsh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,F1 Weighted
1,0.065077,0.018898,0.996155,0.995865,0.995908,0.995859,0.996158
2,0.011790,0.010380,0.997528,0.997378,0.997208,0.997281,0.997528


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.37it/s]
c:\Users\harsh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.71it/s]
c:\Users\harsh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## 15. Evaluate on the untouched test set

In [ ]:
test_metrics = trainer.evaluate(test_dataset)
test_metrics

## 16. Save the best model and tokenizer

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved to:', OUTPUT_DIR.resolve())

## 17. Predict a new support ticket

In [ ]:
def predict_ticket(text, confidence_threshold=0.60):
    model.eval()
    device = next(model.parameters()).device
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(device)
    with torch.inference_mode():
        logits = model(**inputs).logits[0].detach().cpu().numpy()
    probabilities = softmax(logits)
    predicted_id = int(np.argmax(probabilities))
    confidence = float(probabilities[predicted_id])
    return {
        'intent': id2label[predicted_id],
        'confidence': round(confidence, 6),
        'low_confidence': confidence < confidence_threshold,
    }

predict_ticket('My payment was deducted twice and I have not received my refund.')

## Finished

You have now loaded the data, cleaned and split it, created label mappings, used `AutoTokenizer`, tokenized the data, created DistilBERT, written `TrainingArguments`, built a `Trainer`, trained, evaluated, saved, and used the model.